In [1]:
import time
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
import Compocyte
from Compocyte.data import sample_data
from Compocyte.pretrained import til_pretrained

/home/ritwik24222/miniconda3/envs/compo/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import inspect, Compocyte
from Compocyte.core.models.dense_torch import resolve_device
from Compocyte.core.models import fit_methods
print(Compocyte.__file__)
print("device:", resolve_device())
print(inspect.signature(fit_methods.predict))

/home/ritwik24222/Compocyte_walle_lab/src/Compocyte/__init__.py
device: cpu
(model, x, threshold=-1, monte_carlo: int = None, mc_dropout_p: float = 0.5, batch_size: int = 8192, mc_max_rows: int = 65536, device=None)


/home/ritwik24222/miniconda3/envs/compo/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12000). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
base = sample_data()
base

AnnData object with n_obs × n_vars = 861 × 1011
    obs: 'labels'

In [4]:
base = sample_data()
target = 120000
reps = int(np.ceil(target / base.n_obs))
X = sparse.vstack([base.X] * reps, format="csr")[:target]
idx = np.tile(np.arange(base.n_obs), reps)[:target]
obs = base.obs.iloc[idx].copy()
obs.index = [f"large_cell_{i:06d}" for i in range(X.shape[0])]
atlas = sc.AnnData(X=X, obs=obs, var=base.var.copy())
print(atlas.shape)
print(atlas.obs["labels"].value_counts().head())

(120000, 1011)
labels
CD4-T-naive                17220
CD4-TCM                    17220
CD4-TEM                    17172
CD8-T-KLRG1pos-effector    17097
CD8-T-naive                17097
Name: count, dtype: int64


In [5]:
t0 = time.perf_counter()
hc = til_pretrained()
t1 = time.perf_counter()
hc.load_adata(atlas)
t2 = time.perf_counter()
hc.predict_all_child_nodes(hc.root_node)
t3 = time.perf_counter()
total = t3 - t0
n = atlas.n_obs
print(f"load_model_s: {t1 - t0:.1f}")
print(f"load_adata_s: {t2 - t1:.1f}")
print(f"predict_s: {t3 - t2:.1f}")
print(f"total_s: {total:.1f}")
print(f"cells_per_s: {n / total:.1f}")
pred_cols = [c for c in hc.adata.obs.columns if c.endswith("_pred")]
print(pred_cols)
print(hc.adata.obs[pred_cols[-1]].value_counts().head(10))

Neither graph nor dict_of_cell_relations defined upon initialization.
Please run .load() to load an existing classifier.
Predicting at blood.
Predicting at leuko.
Predicting at gran.
Predicting at DC.
Predicting at Langerhans.
Predicting at cDC.
Predicting at mono.
Predicting at c-mono.
Predicting at Mac.
Predicting at B.
Predicting at GC-B.
Predicting at plasma.
Predicting at B-memory.
Predicting at TNK.
Predicting at T.
Predicting at abT.
Predicting at CD8-T.
Predicting at CD8-T-KLRG1pos-effector.
Predicting at CD8-TRM.
Predicting at CD4-T.
Predicting at CD4-TEM.
Predicting at CD4-TRM.
Predicting at CD4-TCM.
Predicting at MAIT.
Predicting at CD8-MAIT.
Predicting at ILC.
Predicting at NK.
load_model_s: 0.2
load_adata_s: 5.0
predict_s: 88.9
total_s: 94.1
cells_per_s: 1275.0
['Level_1_pred', 'Level_2_pred', 'Level_4_pred', 'Level_5_pred', 'Level_3_pred', 'Level_6_pred', 'Level_7_pred']
Level_7_pred
                                        46953
CD4-TCM_nonexhausted                    324